In [63]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [65]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.optim import AdamW, SGD

In [97]:
from src.training.centralized import centralized_train_loop, centralized_validate_loop
from src.data_utils import create_dataloader, read_in_data
from src.models.MatrixFactorization import MF
from src.training.train_utils import EarlyStopper

In [99]:
train_df = pd.read_csv("dataset/ml100k_train_seed10.csv")
test_df = pd.read_csv("dataset/ml100k_test_seed10.csv")
train_df.head()
n_users, n_items, _ = train_df.nunique()
n_users
#train_df, val_df = train_test_split(train_df, test_size=.2, stratify=train_df.user_id, random_state=1)

943

In [101]:
train_df.nunique()

user_id     943
item_id    1633
rating        5
dtype: int64

In [103]:
train_dl = create_dataloader(train_df, dl_type = "centralized", batch_size=10)
val_dl = create_dataloader(val_df, dl_type = "centralized", batch_size=10)
test_dl = create_dataloader(test_df, dl_type = "centralized", batch_size=10)

In [105]:
model = MF(n_users=n_users, n_items=n_items, n_factors=30)

In [107]:
# optimizer = AdamW(model.parameters(), lr=1e-3)
optimizer = SGD(model.parameters(), lr=1e-2, weight_decay=.01, momentum=.1)
early_stopper = EarlyStopper()

In [109]:
centralized_validate_loop(model, val_dl, optimizer)

6.798678572220424

In [81]:
n_users

943

In [91]:
n_epochs = 20
val_losses = []
for epoch in range(n_epochs):
    train_loss = centralized_train_loop(model, train_dl, optimizer)
    val_loss = centralized_validate_loop(model, test_dl, optimizer)
    print(f"Epoch: {epoch+1}, Val loss: {val_loss}")
    if early_stopper.early_stop(val_loss):
        print("Stopping Early.")
        break
    val_losses.append(np.sqrt(val_loss))

  0%|          | 0/7500 [00:00<?, ?it/s]

Epoch: 1, Val loss: 2.823744535285108


  0%|          | 0/7500 [00:00<?, ?it/s]

Epoch: 2, Val loss: 2.4676880585653107


  0%|          | 0/7500 [00:00<?, ?it/s]

...

Epoch: 6, Val loss: 2.391393056553223


  0%|          | 0/7500 [00:00<?, ?it/s]

Epoch: 7, Val loss: 2.3904923048587223


  0%|          | 0/7500 [00:00<?, ?it/s]

Epoch: 8, Val loss: 2.390599033273161
Stopping Early.


In [93]:
val_losses

[1.6804001116654057,
 1.5708876658008715,
 1.5522967401471166,
 1.5475930640815263,
 1.5465200175394442,
 1.5464129644287203,
 1.5461216979457737]